# 08 - MCP Server Integration

> **When to use**: When you want AI assistants (Claude Desktop, Cursor) to directly operate the database to generate test data.
>
> **Core concept**: MCP (Model Context Protocol) lets AI assistants directly invoke sqlseed via 3 tools.

## Applicable Scenarios

- Use natural language to let AI generate test data → MCP Server
- AI assistant needs to view database schema → `sqlseed_inspect_schema`
- AI assistant needs to generate config → `sqlseed_generate_yaml`
- AI assistant needs to execute fill → `sqlseed_execute_fill`

## What You Will Learn

- MCP Server installation and configuration
- Usage of the 3 MCP tools
- Security validation mechanism
- Resource URI access

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| **→ 08** | **MCP Server Integration** | **Plugins: MCP** | **07** |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| MCP Resources and Tools | `plugins/mcp-server-sqlseed/src/mcp_server_sqlseed/server.py` | `mcp` |

> Corresponding architecture diagram: [§10 MCP Server Architecture](../docs/architecture.zh-CN.md#10-mcp-服务器架构)

## 1. See It in Action — AI Assistant Directly Operates the Database

MCP (Model Context Protocol) lets AI assistants (Claude Desktop, Cursor) directly invoke sqlseed — **no manual code needed**:

```
You: "Analyze the projects table structure in app.db, generate YAML config, fill 5000 rows of data"
AI: [auto-calls inspect_schema → generate_yaml → execute_fill]
```

Below we demo local invocations of the 3 MCP tools.

## 2. Why Need an MCP Server?

MCP (Model Context Protocol) lets AI assistants (Claude Desktop, Cursor, etc.) directly operate the database:

- **inspect**: AI views schema, understands table structure
- **generate_yaml**: AI generates data config based on schema
- **execute_fill**: AI executes fill, generates test data

No need to hand-write code or CLI commands — the AI assistant does it all in one stop.

## 3. Claude Desktop / Cursor Configuration

Add the sqlseed server to the AI assistant's MCP config:

In [2]:
import json

config = {
    'mcpServers': {
        'sqlseed': {
            'command': 'python',
            'args': ['-m', 'mcp_server_sqlseed'],
            'env': {'OPENROUTER_API_KEY': 'your-key-here'}
        }
    }
}

print('Claude Desktop / Cursor MCP config:')
print(json.dumps(config, indent=2))

Claude Desktop / Cursor MCP config:
{
  "mcpServers": {
    "sqlseed": {
      "command": "python",
      "args": [
        "-m",
        "mcp_server_sqlseed"
      ],
      "env": {
        "OPENROUTER_API_KEY": "your-key-here"
      }
    }
  }
}


## 4. Tool 1: sqlseed_inspect_schema

AI assistants call this tool to view the database schema, returning column info, FKs, indexes, and sample data.

In [3]:
import json

from mcp_server_sqlseed.server import sqlseed_inspect_schema

result = sqlseed_inspect_schema(str(db_path), table_name='organizations')
print('sqlseed_inspect_schema result (dict):')
print(json.dumps(result, indent=2, ensure_ascii=False)[:500])

sqlseed_inspect_schema result (dict):
{
  "organizations": {
    "table_name": "organizations",
    "columns": [
      {
        "name": "org_code",
        "type": "VARCHAR(16)",
        "nullable": false,
        "default": null,
        "is_primary_key": true,
        "is_autoincrement": false
      },
      {
        "name": "name",
        "type": "VARCHAR(64)",
        "nullable": false,
        "default": null,
        "is_primary_key": false,
        "is_autoincrement": false
      },
      {
        "name": "parent_code",
 


## 5. Tool 2: sqlseed_generate_yaml

AI analyzes schema and generates a YAML config. Requires API Key.

In [4]:
import os

api_key = os.environ.get('OPENROUTER_API_KEY') or os.environ.get('OPENAI_API_KEY')

if api_key:
    from mcp_server_sqlseed.server import sqlseed_generate_yaml
    result = sqlseed_generate_yaml(str(db_path), table_name='organizations', api_key=api_key)
    print('sqlseed_generate_yaml result:')
    print(result[:500] if isinstance(result, str) else str(result)[:500])
else:
    print('No API key. Example AI-generated YAML:')
    print('''tables:
  - name: organizations
    count: 100
    columns:
      - name: org_code
        generator: pattern
        params:
          pattern: "ORG-\\d{4}"
      - name: name
        generator: company
      - name: description
        generator: sentence
        params:
          nb_words: 8''')

sqlseed_generate_yaml result:
db_path: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db
provider: mimesis
locale: en_US
tables:
- name: organizations
  count: 1000
  columns:
  - name: org_code
    generator: pattern
    params:
      regex: ORG-[0-9]{4,6}
  - name: name
    generator: company
  - name: parent_code
    generator: foreign_key
    params:
      ref_table: organizations
      ref_column: org_code
  - name: description
    generator: text
    params:
      min_length: 50
      max_length: 200
  -


## 6. Tool 3: sqlseed_execute_fill

Execute data fill, supports inline YAML config.

In [5]:
import json

from mcp_server_sqlseed.server import sqlseed_execute_fill

result = sqlseed_execute_fill(str(db_path), table_name='tags', count=3)
print('sqlseed_execute_fill result:')
print(json.dumps(result, indent=2, ensure_ascii=False)[:500])

Generating tags:   0%|          | 0/3 [00:00<?, ?it/s]

sqlseed_execute_fill result:
{
  "table_name": "tags",
  "count": 3,
  "elapsed": 0.024562137987231836,
  "errors": []
}


## 7. Security Validation

MCP Server has built-in security validation to prevent path traversal and SQL injection.

In [6]:
from mcp_server_sqlseed.server import _validate_db_path, _validate_table_name

print('Security validation functions:')
print()

# Valid path (existing .db file)
try:
    result = _validate_db_path(str(db_path))
    print(f'  _validate_db_path("{db_path.name}") -> OK (resolved: {result})')
except Exception as e:
    print(f'  _validate_db_path("{db_path.name}") -> {type(e).__name__}: {e}')

# Invalid paths
for path in ['../../../etc/passwd', 'test.txt', 'nonexistent.db']:
    try:
        _validate_db_path(path)
        print(f'  _validate_db_path("{path}") -> OK')
    except Exception as e:
        print(f'  _validate_db_path("{path}") -> {type(e).__name__}: {e}')

# Table name validation (requires allowed_tables list)
allowed = ['organizations', 'members', 'projects', 'tasks', 'tags']
for name in ['organizations', '; DROP TABLE --', '../hack']:
    try:
        _validate_table_name(name, allowed)
        print(f'  _validate_table_name("{name}") -> OK')
    except Exception as e:
        print(f'  _validate_table_name("{name}") -> {type(e).__name__}: {e}')

Security validation functions:

  _validate_db_path("sqlseed_demo.db") -> OK (resolved: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db)
  _validate_db_path("../../../etc/passwd") -> ValueError: Invalid database path: ../../../etc/passwd. Must be a .db, .sqlite, or .sqlite3 file.
  _validate_db_path("test.txt") -> ValueError: Invalid database path: test.txt. Must be a .db, .sqlite, or .sqlite3 file.
  _validate_db_path("nonexistent.db") -> ValueError: Database file not found: nonexistent.db
  _validate_table_name("organizations") -> OK
  _validate_table_name("; DROP TABLE --") -> ValueError: Table '; DROP TABLE --' does not exist in the database. Available: ['organizations', 'members', 'projects', 'tasks', 'tags']
  _validate_table_name("../hack") -> ValueError: Table '../hack' does not exist in the database. Available: ['organizations', 'members', 'projects', 'tasks', 'tags']


## Summary

| MCP Tool | Function | Requires API Key |
|----------|------|:------------:|
| `sqlseed_inspect_schema` | View schema | ❌ |
| `sqlseed_generate_yaml` | AI generates config | ✅ |
| `sqlseed_execute_fill` | Execute fill | ❌ |

**Next**: [09-plugin-hooks.ipynb](09-plugin-hooks.ipynb) — Plugin System and Hook Lifecycle

In [7]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
